# CARE Is a 4.2x Multiplier: Animal Economics

Most Kaggriculture strategy talk is about crops. This notebook is about the animal side,
where I think the single largest per-tile lever in the game is hiding in plain sight.

**TL;DR**

- `CARE` does not add a flat bonus. It changes an animal's output from **1 unit per
  production cycle to `1 + interval` units**. Measured over a 30-day season by running the
  engine's own refresh function: **2.07x for geese, 3.25x for cows, 4.22x for sheep.**
  The multiplier *grows* with the interval, so the slowest animal gains the most.
- At base prices, one cared-for sheep-day is worth **$241.67 net of feed** against
  **$75.00** for a goose-day — **3.2x**, for the same tile and the same two daily actions.
- CARE decides **which** animal you keep. The glut curve decides **how many**. Scaling one
  species past its town sink destroys the value CARE created, and I have both a model and
  288 paired episodes that say so.
- Part 6 ships a **working 130-line husbandry bot** (written to `main.py`, fork and submit
  it) plus the ablation that flips one line: **CARE on beats CARE off on 12 of 12 identical
  seeds**, 7.12x mean final bank.

Everything in Parts 1, 2 and 3 is recomputed live from the installed engine when you run
this notebook — I would rather you check me than believe me.

*I'm a high-school student doing this competition as an economics project. If something
here is wrong, please say so in the comments.*

In [ ]:
# --- Load the shipped engine. Fail LOUDLY rather than quietly guessing. ---
import inspect
import math
import subprocess
import sys


def _version():
    try:
        import importlib.metadata as _md
        return _md.version("kaggle-environments")
    except Exception:
        return "unknown"

def _load():
    for mod in [m for m in list(sys.modules) if m.startswith("kaggle_environments")]:
        del sys.modules[mod]
    from kaggle_environments.envs.kaggriculture import kaggriculture as K
    return K

# The Kaggle image ships kaggle-environments 1.29.3, which is PRE-REBALANCE: it still has
# the old town-centre demand schedule. Every number below would silently describe a game
# that no longer exists, so upgrade first and say so out loud.
BEFORE = _version()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "kaggle-environments"], check=False)
try:
    K = _load()
except Exception as e:
    sys.exit(f"FATAL: could not import the kaggriculture engine: {e!r}")
if BEFORE != _version():
    print(f"upgraded kaggle-environments {BEFORE} -> {_version()}")

for required in ("ANIMALS", "MARKET_PARAMS", "market_price",
                 "_new_animal", "_daily_refresh_animals"):
    if not hasattr(K, required):
        sys.exit(f"FATAL: engine is missing {required!r} — this notebook's "
                 f"assumptions do not hold on this version. Stopping.")

print(f"kaggle-environments == {_version()}")
print("The prose in this notebook was written against 1.32.6.")

# The prose quotes specific constants. If the engine has moved under us, say so
# in capital letters instead of letting the text and the tables disagree.
EXPECTED = {
    ("ANIMALS", "GOOSE", "interval"): 1, ("ANIMALS", "COW", "interval"): 2,
    ("ANIMALS", "SHEEP", "interval"): 3,
    ("MARKET_PARAMS", "EGG", "base"): 50, ("MARKET_PARAMS", "MILK", "base"): 160,
    ("MARKET_PARAMS", "WOOL", "base"): 200, ("MARKET_PARAMS", "WHEAT", "base"): 25,
}
drift = []
for (table, key, field), want in EXPECTED.items():
    got = getattr(K, table)[key].get(field)
    if got != want:
        drift.append(f"  {table}[{key}][{field}]: prose says {want}, engine says {got}")
if drift:
    print("\n*** WARNING: THE ENGINE HAS CHANGED SINCE THIS WAS WRITTEN ***")
    print("\n".join(drift))
    print("Trust the computed tables below, NOT the prose numbers.")
else:
    print("Constant check: OK — every number quoted in the prose still matches the engine.")


---

## Part 1 — What CARE actually does

Here is the mechanic, straight out of the engine. I am printing the real source rather
than paraphrasing it, because the ordering inside this function is the whole finding.

In [ ]:
src = inspect.getsource(K._daily_refresh_animals)
# Print the production/care block itself — the ordering inside it is the finding.
marker = "a = ANIMALS[tile[\"animal\"]]"
print(src[src.index(marker):] if marker in src else src)


Read the order carefully:

1. On a **production day**, if the animal was fed, the engine pays out the *entire*
   accumulated bonus at once — `yield_units += 1 + bonus` — and then resets the counter
   to zero.
2. **After** that, if the animal was both fed and cared for today, the counter goes up
   by one.

The counter is only ever emptied on a production day. So it accumulates across the whole
interval, and the steady-state output per production cycle is:

```
units per cycle = 1 + interval
```

That is the counter-intuitive part. `CARE` is not a percentage bonus; it is **+1 unit per
cared-for day**, banked and paid on the next production tick. A goose produces every day,
so it can only ever bank one day of care. A sheep produces every third day, so it banks
three. **The slower the animal, the more CARE is worth.**

In [ ]:
for name, a in K.ANIMALS.items():
    print(f"{name:6s} interval={a['interval']}  cost=${a['cost']:<4d} "
          f"first_yield_day={a['first_yield_day']:<2d} max_held={a['max_held']}  "
          f"product={a['product']:5s}  ->  steady state {1 + a['interval']} units/cycle")


### Don't take my word for the steady state — drive the engine

The function below builds a real animal tile with the engine's own `_new_animal`, then
calls the engine's own `_daily_refresh_animals` once per day for a 30-day season. It feeds
the animal every day, cares for it (or doesn't), and collects whenever units are sitting
on the tile. No re-implementation of the rules, no hand algebra.

In [ ]:
SEASON_DAYS = 30       # episodeSteps 720 / turnsPerDay 24

def run_animal(animal, care=True, season=SEASON_DAYS, buy_day=0):
    """Drive the engine's own daily refresh. Returns total units produced."""
    tile = K._new_animal(animal, buy_day)
    farm = {"tiles": [[tile]]}
    produced = 0
    for day in range(buy_day, season):
        tile["fed_today"] = True            # FEED  (costs 1 wheat)
        tile["cared_today"] = bool(care)    # CARE  (free, one action)
        if tile["yield_units"] > 0:         # COLLECT
            produced += tile["yield_units"]
            tile["yield_units"] = 0
        K._daily_refresh_animals(farm, day)
    return produced + tile["yield_units"]

print(f"{'animal':7s} {'w/ CARE':>8s} {'no CARE':>8s} {'multiplier':>11s}")
care_units = {}
for name in K.ANIMALS:
    w, wo = run_animal(name, care=True), run_animal(name, care=False)
    care_units[name] = w
    print(f"{name:7s} {w:8d} {wo:8d} {w / wo:10.2f}x")


In [ ]:
import matplotlib.pyplot as plt

names = list(K.ANIMALS)
w  = [run_animal(n, care=True)  for n in names]
wo = [run_animal(n, care=False) for n in names]

fig, ax = plt.subplots(figsize=(7, 3.6))
x = range(len(names))
ax.bar([i - 0.2 for i in x], wo, 0.4, label="fed only", color="#c9ccd1")
ax.bar([i + 0.2 for i in x], w,  0.4, label="fed + CARE", color="#2e7d32")
for i, (a, b) in enumerate(zip(wo, w)):
    ax.text(i + 0.2, b + 1, f"{b/a:.2f}x", ha="center", fontweight="bold")
ax.set_xticks(list(x)); ax.set_xticklabels(names)
ax.set_ylabel("units produced in a 30-day season")
ax.set_title("CARE pays more the slower the animal")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()


---

## Part 2 — What that is worth per animal-day

Now price it. Feeding costs exactly one `WHEAT` per animal per day — that is the whole
cost of the `FEED` op — and wheat's base price is $25, so every animal-day carries a $25
opportunity cost whether you bought the wheat or grew it.

In [ ]:
WHEAT_BASE = K.MARKET_PARAMS["WHEAT"]["base"]

print(f"feed cost = 1 WHEAT/animal/day = ${WHEAT_BASE}/day\n")
print(f"{'animal':7s} {'product':6s} {'base':>6s} {'units/cycle':>12s} "
      f"{'gross/day':>10s} {'net of feed':>12s}")
for name, a in K.ANIMALS.items():
    base = K.MARKET_PARAMS[a["product"]]["base"]
    per_cycle = 1 + a["interval"]
    gross = per_cycle * base / a["interval"]
    print(f"{name:7s} {a['product']:6s} ${base:5d} {per_cycle:12d} "
          f"${gross:9.2f} ${gross - WHEAT_BASE:11.2f}")


A sheep-day is worth **3.2x** a goose-day net of feed, on the same tile, for the same two
daily actions (`FEED` + `CARE`).

This is where I got the game wrong at first. I opened by building geese, because egg has
by far the friendliest glut curve — you will see it in Part 3 — and I reasoned that an
unbounded price beats a high price. That reasoning was fine and the conclusion was still
wrong, because I had not priced the CARE multiplier. A per-unit price advantage is
worthless if you have mispriced the production multiplier that sits in front of it.

---

## Part 3 — Why you cannot just buy more

Part 2 values one animal in a vacuum. A herd is a different question, because the price of
what you sell is a function of how much of it exists. Here is the engine's own
`market_price` against inventory, for the three animal products.

In [ ]:
I0 = K.MARKET_I0
print(f"{'dumped':>7s} {'EGG':>6s} {'MILK':>6s} {'WOOL':>6s}")
for n in (0, 25, 50, 100, 200, 400, 800):
    row = "".join(f"{K.market_price(p, I0 + n):6d} " for p in ("EGG", "MILK", "WOOL"))
    print(f"{n:7d} {row}")

print()
for p in ("EGG", "MILK", "WOOL"):
    mp = K.MARKET_PARAMS[p]
    print(f"{p:5s} glut shape={mp['above_func']:6s} target={mp['above_target']:.2f} T={mp['T']}")


`EGG` uses a logarithmic glut curve with a target of only 0.20 over a capacity of 332 —
it is nearly flat, and still pays $38 after eight hundred units. `WOOL` uses a **squared**
curve with a target of 3.20 over a capacity of just 105. It falls off a cliff.

So the two halves of the animal decision point in opposite directions: CARE says sheep,
the glut curve says egg. The resolution is the town sink — the town permanently removes
inventory, which is what lets a price recover — so what matters is your herd's output
*relative to the sink for that one product*.

The next cell is a **model**, not an engine fact. It spreads each animal's production
evenly across its active days, assumes your opponent sells the same amount you do, and
unlocks town shops on the real schedule (one every 3 days, capped at 8 instances, drawn
with replacement). Those assumptions are all arguable — they are in the code so you can
argue with them.

In [ ]:
UNLOCK_INTERVAL, MAX_SHOPS, TICKS_PER_DAY = 3, 8, 6

def sink_on_day(product, day):
    """Expected units/day the town permanently removes."""
    n = min(MAX_SHOPS, day // UNLOCK_INTERVAL)
    rate = sum(n * (1.0 / len(K.SHOPS)) * (2 if len(ps) == 1 else 1)
               for s, ps in K.SHOPS.items() if product in ps)
    rate *= TICKS_PER_DAY
    return rate + (1.0 if product in K.TOWN_CENTER_PRODUCTS else 0.0)

def realized_revenue(animal, herd, season=SEASON_DAYS, opponent=True):
    """Sell the herd's output day by day into the live price curve. MODEL."""
    a = K.ANIMALS[animal]
    product = a["product"]
    per_animal = care_units[animal]
    start = a["first_yield_day"]
    per_day = per_animal / (season - start) * herd
    inv, total, mine, carry = float(I0), 0.0, 0, 0.0
    share = 2 if opponent else 1
    for day in range(season):
        if day >= start:
            carry += per_day * share
        k, carry = int(carry), carry - int(carry)
        for i in range(k):
            if i % share == 0:                      # our half of the flow
                total += K.market_price(product, int(round(inv)))
                mine += 1
            inv += 1
        inv = max(I0 - 500, inv - sink_on_day(product, day))
    return total, mine

print(f"{'herd':>5s} " + " ".join(f"{a:>14s}" for a in K.ANIMALS))
print(f"{'':5s} " + " ".join(f"{'$/animal':>14s}" for _ in K.ANIMALS))
for herd in (1, 2, 4, 6, 8, 12, 16):
    cells = []
    for animal in K.ANIMALS:
        rev, _ = realized_revenue(animal, herd)
        cells.append(f"{rev / herd:14,.0f}")
    print(f"{herd:5d} " + " ".join(cells))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

xs = range(0, 401, 5)
for prod, colour in (("EGG", "#f9a825"), ("MILK", "#1e88e5"), ("WOOL", "#8e24aa")):
    ax1.plot(list(xs), [K.market_price(prod, I0 + n) for n in xs], label=prod, color=colour)
ax1.set_xlabel("units dumped into the market"); ax1.set_ylabel("price ($)")
ax1.set_title("The glut curve: wool falls off a cliff, egg barely moves")
ax1.legend(frameon=False); ax1.spines[["top", "right"]].set_visible(False)

herds = [1, 2, 4, 6, 8, 12, 16]
for animal, colour in (("GOOSE", "#f9a825"), ("COW", "#1e88e5"), ("SHEEP", "#8e24aa")):
    ys = [realized_revenue(animal, h)[0] / h for h in herds]
    ax2.plot(herds, ys, marker="o", label=animal, color=colour)
ax2.set_xlabel("herd size (one species)"); ax2.set_ylabel("revenue per animal ($)")
ax2.set_title("...so the value of the marginal animal collapses  [MODEL]")
ax2.legend(frameon=False); ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout(); plt.show()


Geese barely decay: about $3,050 per animal at a herd of one, still about $2,200 at
sixteen. Cows and sheep start two to three times higher and then collapse — a sheep is
worth roughly $11,100 alone and about $370 in a flock of sixteen, a **30x** fall. The
crossover, where a goose would have been the better animal, sits around a herd of four
to six of a single species.

That is the trap. The CARE table in Part 2 is real, and it will happily talk you into
filling the board with sheep, which is the one thing that makes the CARE table stop being
true.

---

## Part 4 — 288 paired episodes say the same thing

The model above is only a model, so I also ran it as an experiment against my own agent.
Each configuration played the **same 288 seeds** as the baseline, both seatings, against
the same opponents — so the pairs are matched and the right test is McNemar on the
discordant pairs, not a two-proportion z. (An unpaired z here would be anti-conservative
and would overstate every one of these.)

The baseline agent runs `target_cows=8`, `target_sheep=6`, `target_geese=0`,
`care_mult=0.35` (a weight on how eagerly it spends actions on `CARE`).

**These episodes were run locally; the win/loss counts below are transcribed from that
run, not recomputed here.** The statistic is computed live so you can at least check the
arithmetic.

In [ ]:
# label -> (overall win %, W, L)  — W/L are DISCORDANT pairs vs baseline, n=288 each.
SWEEP = [
    ("baseline (cows 8, sheep 6, care 0.35)", 19.1, None, None),
    ("care_mult 0.35 -> 0.20  (care less)",   10.1,  4, 30),
    ("target_sheep 6 -> 8",                    8.7,  7, 37),
    ("target_sheep 6 -> 4",                    9.7,  6, 33),
    ("target_cows  8 -> 10",                  12.5,  5, 24),
    ("target_cows  8 -> 6",                   18.4, 17, 19),
]

print(f"{'change':40s} {'win%':>6s} {'W/L':>8s} {'z':>7s}  verdict")
for label, win, W, L in SWEEP:
    if W is None:
        print(f"{label:40s} {win:5.1f}% {'—':>8s} {'—':>7s}")
        continue
    z = (W - L) / math.sqrt(W + L)
    verdict = ("significantly WORSE" if z <= -1.96 else
               "significantly better" if z >= 1.96 else "no detectable effect")
    print(f"{label:40s} {win:5.1f}% {W:3d}/{L:<4d} {z:+7.2f}  {verdict}")


Three things fall out, and they are the same three things the model predicted:

- **Caring less is significantly worse** (z = −4.46). CARE is not a rounding error; cutting
  the agent's willingness to spend actions on it costs nearly half the win rate.
- **Scaling either species up is significantly worse** — sheep 6→8 at z = −4.52, cows 8→10
  at z = −3.53. This is the glut curve arriving exactly where Part 3 said it would.
- **Scaling cows down is free** (z = −0.33). Eight cows was already past the point where the
  extra animals were paying for themselves.

Note what is *not* here: no configuration in this sweep was significantly **better** than
baseline. That is the honest result, and it is worth more to you than a fake win.

---

## Part 5 — What to actually do

The single sentence I would take away:

> **CARE decides which animal you keep. The town sink decides how many.**

Concretely, from the numbers above:

1. **Care for every animal, every day, before you consider buying another one.** A cared-for
   sheep is 4.22x an uncared one. No herd-size decision in this game is worth as much as
   turning that multiplier on, and it costs one action and zero currency.
2. **Buying an animal you cannot also feed and care for is worse than not buying it.**
   An uncared sheep produces 9 units a season. Two of them cost $1,000, two tiles and
   double the feed to produce 18 — while one cared sheep produces 38 for half of everything.
3. **Diversify across products, not within one.** This is the bit the model makes obvious.
   Each product has its own inventory and its own price curve, so a mixed herd of cows and
   sheep sells into *two* independent markets, while the same number of sheep sells into
   one. That is why my baseline's mixed 8-cows-plus-6-sheep survives at a herd size where
   Part 3's single-species curve has already collapsed.
4. **Do not read Part 2 as "sheep are the best animal".** Read it as "the marginal animal is
   worth far less than the first one, and CARE is worth more than either".

Here is the calculator, so you can put your own assumptions in.

In [ ]:
def animal_plan(budget, tiles, season=SEASON_DAYS, verbose=True):
    """Rank single-species herds you can afford. MODEL — see Part 3's assumptions."""
    rows = []
    for animal, a in K.ANIMALS.items():
        max_by_cash = budget // a["cost"]
        herd = int(min(tiles, max_by_cash))
        if herd < 1:
            continue
        rev, units = realized_revenue(animal, herd, season)
        feed = WHEAT_BASE * herd * (season - a["first_yield_day"])
        capital = a["cost"] * herd
        rows.append((rev - feed - capital, animal, herd, rev, units, feed, capital))
    rows.sort(reverse=True)
    if verbose:
        print(f"budget ${budget:,}  tiles {tiles}  season {season}d\n")
        print(f"{'animal':7s} {'herd':>5s} {'units':>7s} {'revenue':>10s} "
              f"{'feed':>8s} {'capital':>8s} {'PROFIT':>10s}")
        for profit, animal, herd, rev, units, feed, capital in rows:
            print(f"{animal:7s} {herd:5d} {units:7.0f} {rev:10,.0f} "
                  f"{feed:8,.0f} {capital:8,.0f} {profit:10,.0f}")
    return rows

animal_plan(budget=3000, tiles=6)
print()
animal_plan(budget=12000, tiles=16)


---

## Part 6 — A bot that does only this, and the ablation

Everything above is analysis. Here is a working agent that does nothing except keep a
mixed herd of 4 cows and 3 sheep fed, cared for and harvested. It grows no crops, runs no
market model, hires three hands and drip-sells. It is about 130 lines.

It is written to `main.py`, so you can fork this notebook, run the cell, and submit it —
or, more usefully, lift the worker-priority block into your own agent.

In [ ]:
%%writefile main.py
"""CARE-first animal husbandry agent.

A deliberately small agent that does one thing: keep a mixed herd fed, cared for and
harvested every single day. It grows no crops and runs no clever market model.
"""

SHED_TILES = [(4, 4), (5, 4), (4, 5), (5, 5)]
HERD = [("COW", 4), ("SHEEP", 3)]          # two species on purpose: two price curves
# NW is the only quadrant unlocked at the start, so every spot must satisfy x<5 and y<5.
# Ordered by walking distance from the shed-access tile (4,4).
PASTURE_SPOTS = [(4, 3), (3, 4), (3, 3), (4, 2), (2, 4), (2, 3), (3, 2), (4, 1)]
PRODUCTS = ("MILK", "WOOL", "EGG", "FERTILIZER")
CARE_ENABLED = True   # flip to False for the ablation


def _step(pos, target):
    x, y = pos
    tx, ty = target
    if x < tx: return "EAST"
    if x > tx: return "WEST"
    if y < ty: return "SOUTH"
    if y > ty: return "NORTH"
    return None


def _dist(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def agent(obs):
    me = obs["farms"][obs["player"]]
    priv = obs["private"]
    tiles = me["tiles"]
    shed = priv["shed"]
    money = me["money"]
    invs = priv["inventories"]

    placed = {}
    empty_pastures = []
    animal_tiles = []
    free_spots = []
    for y, row in enumerate(tiles):
        for x, t in enumerate(row):
            if isinstance(t, dict) and "animal" in t:
                placed[t["animal"]] = placed.get(t["animal"], 0) + 1
                animal_tiles.append(((x, y), t))
            elif isinstance(t, dict) and t.get("kind") == "PASTURE":
                empty_pastures.append((x, y))
            elif t is None and (x, y) in PASTURE_SPOTS:
                free_spots.append((x, y))

    n_target = sum(n for _, n in HERD)
    n_structures = len(empty_pastures) + sum(placed.values())
    shed_animals = sum(shed.get(a, 0) for a, _ in HERD)

    # ---------------- market ----------------
    market = []
    carried_wheat = sum(i.get("WHEAT", 0) for i in invs)
    total_wheat = shed.get("WHEAT", 0) + carried_wheat
    n_alive = sum(placed.values())
    # keep roughly three days of feed in stock
    if total_wheat < max(8, n_alive * 3) and money > 600:
        market.append(["BUY_PRODUCT", "WHEAT", 12])

    for animal, target in HERD:
        if placed.get(animal, 0) + shed.get(animal, 0) < target and money > 1400:
            market.append(["BUY_ANIMAL", animal, 1])
            break

    for product in ("MILK", "WOOL"):
        held = shed.get(product, 0)
        if held > 0:
            market.append(["SELL", product, min(held, 4)])   # drip, never dump

    if obs["day"] >= 1 and me["hires_today"] < 3 and money > 2000:
        market.append(["HIRE"])

    # ---------------- workers ----------------
    workers = [tuple(me["farmer"])] + [tuple(p) for p in me["hands"]]
    claimed = set()
    ops = []

    for wi, pos in enumerate(workers):
        inv = invs[wi] if wi < len(invs) else {}
        x, y = pos
        here = tiles[y][x]
        carrying_animal = next((a for a, _ in HERD if inv.get(a, 0) > 0), None)
        job = None

        # --- act where we stand ---
        if isinstance(here, dict) and "animal" in here:
            if here.get("yield_units", 0) > 0:
                job = ["HARVEST"]
            elif not here.get("fed_today") and inv.get("WHEAT", 0) > 0:
                job = ["FEED"]
            elif CARE_ENABLED and not here.get("cared_today"):
                job = ["CARE"]
        elif carrying_animal and isinstance(here, dict) and here.get("kind") == "PASTURE":
            job = ["PLACE", carrying_animal, 1]
        elif here is None and (x, y) in PASTURE_SPOTS and n_structures < n_target:
            job = ["BUILD_PASTURE"]
        elif pos in SHED_TILES:
            if any(inv.get(p, 0) for p in PRODUCTS):
                job = ["DROP"]
            elif not carrying_animal and shed_animals > 0 and empty_pastures:
                for a, _ in HERD:
                    if shed.get(a, 0) > 0:
                        job = ["PICKUP", a, 1]
                        shed[a] -= 1
                        break
            elif inv.get("WHEAT", 0) < 3 and shed.get("WHEAT", 0) > 0:
                job = ["PICKUP", "WHEAT", 4]

        # --- otherwise, go somewhere useful ---
        if job is None:
            target = None
            if carrying_animal and empty_pastures:
                target = min((p for p in empty_pastures if p not in claimed),
                             key=lambda p: _dist(p, pos), default=None)
            if target is None and not carrying_animal and shed_animals > 0 and empty_pastures:
                target = min(SHED_TILES, key=lambda s: _dist(s, pos))
            if target is None and n_structures < n_target and free_spots:
                target = min((s for s in free_spots if s not in claimed),
                             key=lambda s: _dist(s, pos), default=None)
            if target is None and inv.get("WHEAT", 0) == 0 and shed.get("WHEAT", 0) > 0:
                target = min(SHED_TILES, key=lambda s: _dist(s, pos))
            if target is None:
                best, best_score = None, 0.0
                for (ax, ay), t in animal_tiles:
                    if (ax, ay) in claimed:
                        continue
                    score = 0.0
                    if t.get("yield_units", 0) > 0: score += 3
                    if not t.get("fed_today") and inv.get("WHEAT", 0) > 0: score += 2
                    if CARE_ENABLED and not t.get("cared_today"): score += 1
                    score -= 0.1 * _dist((ax, ay), pos)
                    if score > best_score:
                        best, best_score = (ax, ay), score
                target = best
            if target is None and any(inv.get(p, 0) for p in PRODUCTS):
                target = min(SHED_TILES, key=lambda s: _dist(s, pos))
            if target is not None:
                claimed.add(target)
                mv = _step(pos, target)
                job = [mv] if mv else ["PASS"]
            else:
                job = ["PASS"]

        ops.append(job)

    return {"farmer": ops[0], "hands": ops[1:], "market": market[:10]}


Now the ablation. The **same agent**, on the **same 12 seeds**, against the same opponent,
with one line changed: `CARE_ENABLED = False`. Because the pairs are matched, the honest
count is wins on discordant seeds, not a difference of means.

Note which way this cuts. Switching CARE off *frees an action per animal per day*, which
the agent spends on feeding and harvesting instead. So the control arm gets strictly more
labour, and still loses.

In [ ]:
import os
import statistics
import sys

sys.path.insert(0, os.getcwd())
import main as care_bot
from kaggle_environments import make

SEEDS = list(range(101, 113))

def play(seeds, care):
    care_bot.CARE_ENABLED = care
    banks = []
    for seed in seeds:
        env = make("kaggriculture", configuration={"seed": seed}, debug=False)
        env.run([care_bot.agent, "starter"])
        banks.append(env.state[0].reward)
    return banks

on  = play(SEEDS, True)
off = play(SEEDS, False)

wins = sum(1 for a, b in zip(on, off) if a > b)
losses = sum(1 for a, b in zip(on, off) if a < b)
z = (wins - losses) / math.sqrt(wins + losses) if wins + losses else 0.0

print(f"{'seed':>6s} {'CARE on':>12s} {'CARE off':>12s}")
for s, a, b in zip(SEEDS, on, off):
    print(f"{s:6d} {a:12,.0f} {b:12,.0f}")
print("-" * 34)
print(f"{'mean':>6s} {statistics.mean(on):12,.0f} {statistics.mean(off):12,.0f}")
print(f"\nCARE on wins {wins}/{len(SEEDS)} paired seeds "
      f"(losses {losses}), McNemar z = {z:+.2f}")
print(f"ratio of mean final bank: {statistics.mean(on)/statistics.mean(off):.2f}x")


On `kaggle-environments` **1.32.6** that is **12/12 paired seeds** and a **7.12x** ratio of
mean final bank ($30,402 against $4,269).

> **Check your engine version before you trust any animal number, including mine.**
> The Kaggle image still ships **1.29.3**, which is *pre-rebalance* — it has the old town
> centre consuming twice a day at up to 4x. I ran this same ablation on both: 1.29.3 gives
> **12/12 and 30.53x**, 1.32.6 gives **12/12 and 7.12x**. The direction is identical on both
> engines; the magnitude is off by a factor of four. The first cell of this notebook
> upgrades the package for exactly this reason, and prints the version so you can see which
> game you are looking at.

The cell above recomputes everything, so if your numbers differ from my paragraph, believe
your run.

Two things this is **not**:

- **It is not a leaderboard claim.** The opponent here is the built-in `starter` bot, which
  is very weak — it banks about $3,500. Beating it proves the husbandry loop works, and
  nothing else. My own tuned agents beat this bot by far more and still sit mid-table.
- **7.12x is bigger than the 3-4x mechanic multiplier from Part 1, and that gap is
  compounding, not magic.** More product early means more cash, which means feed and the
  last animals arrive sooner. The clean per-animal number is the one in Part 1.

### An open disagreement I have not resolved

Run that calculator at a large herd and it says **geese**, by a wide margin — $20,174 profit
at sixteen geese against a loss for sixteen sheep. My actual agent runs `target_geese=0`
and I have never found a goose configuration that beat it in paired episodes.

So the model and the measurements disagree, and I am not going to pretend otherwise. The
two candidates I can think of:

- **The model lets you sell too smoothly.** It spreads 907 eggs evenly across the season.
  A real agent produces and dumps in lumps, and a lump walks much further down the price
  curve than a trickle does. If so, the model overstates every high-volume strategy, and
  geese are the highest-volume strategy in the game.
- **Tiles are not the binding constraint; actions are.** The calculator budgets cash and
  tiles. It does not budget farmer-turns, and a sixteen-animal herd needs feeding, caring
  and collecting every single day by farmers who also have to plant, water and sell.

I have not run the experiment that separates these, so I am not claiming either. If you
run it, I would genuinely like to know — that is a more interesting result than anything
above.

---

## Honest limits

I would rather be corrected than upvoted, so here is what this notebook does **not** show.

**Parts 1 and 2 are exact.** They are read off the shipped engine and recomputed when you
run this, and the first cell shouts if the constants have moved.

**Part 3 is a model.** It smooths production evenly across the season, assumes a mirrored
opponent, and approximates the town's shop draws in expectation rather than sampling them.
Real episodes are lumpier than this in every one of those dimensions. The *shape* — egg
flat, wool cliff-edged — is an engine fact; the specific dollar figures are not.

**Part 4 is self-play, and self-play has already lied to me in this competition.** Two of
my own submitted agents were ranked backwards by local win-rate: the one that beat the
other in paired local episodes scored **752.7** on the actual ladder, while the agent it
beat scored **786.6**. So please read Part 4 as evidence about *the mechanic* — CARE pays,
over-scaling one species doesn't — and not as a promise that any particular herd size will
move your rating.

**The engine changed under us mid-competition.** Town demand was rebalanced in 1.32.6.
Everything here is computed from whatever version this notebook runs on, which is why I
print the version and diff the constants at the top. Re-run it; don't trust a screenshot,
including mine.

If you find an error, please leave a comment — that is the most useful thing you can do
with this notebook.